# ITCHI v0.2 por snapshot: Cristóbal 2020 y Odile 2014

El notebook 05 resume cada evento con `ITCHI_max`, el máximo en el tiempo. Eso dice dónde hubo peligro, pero no cuándo, cuánto tiempo duró, ni qué cambio de la v0.2 lo produjo. Aquí se analiza el peligro en cada snapshot de 6 h.

**Cadena de experimentos.** Cada paso cambia una sola cosa, para atribuir las diferencias entre v0.1 y v0.2:

| Experimento | Lluvia $H_P$ | Viento $H_W$ | Paso respecto al anterior |
|---|---|---|---|
| `v0.1` | Rampa R3 | Rankine, solo dentro de R34 | — |
| `R4` | Rampa R4 | Igual | rampa R4 (alerta desde P75) |
| `R4+esc` | R4 escalada por R24 y A72 | Igual | escalada |
| `W06 sin cola` | R4 escalada | W06 calibrado, solo dentro de R34 | viento W06 dentro de R34 |
| `v0.2` | R4 escalada | W06 con cola hasta R_W06 | cola de viento (componente indirecto) |

`W06 sin cola` no se corre aparte: sale de los campos de la v0.2 quitando el viento del componente indirecto, $\mathrm{ITCHI} = 1-(1-H_{dir})(1-H_P\,M_{ind})$.

**Qué se calcula.**
1. Mapas de clase SIAT en snapshots elegidos y mapa de la causa de cada cambio.
2. Series por snapshot: ITCHI máximo, área en tierra por clase y municipios en alerta.
3. Verificación por snapshot: PSS instantáneo y acumulado contra las declaratorias del ciclón.
4. Atribución por paso: qué paso de la cadena sube o baja la clase municipal y cómo cambia el PSS.
5. Duración y oportunidad de la alerta por municipio.

**Cuidado con las muestras.** Cristóbal tiene 125 municipios con declaratoria. Odile solo tiene 6, así que sus POD y PSS cambian mucho con un solo municipio. Odile ilustra, pero no sirve para comparar experimentos.

Los campos se guardan en `outputs/diagnostics/itchi_v02_snapshots/`: la segunda corrida no recalcula nada. Las figuras van a `outputs/figures/itchi_v02_snapshots/`.

## 0. Configuración

In [ ]:
import os
import re
import time
import warnings
from pathlib import Path

import cartopy.crs as ccrs
import geopandas as gpd
import matplotlib.dates as mdates
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import xarray as xr
from matplotlib.colors import BoundaryNorm, ListedColormap
from matplotlib.patches import Patch

import itchi.itchi_size_helpers as H
from itchi.config import get_project_root, load_default_config
from itchi.constants import ITCHI_VERSION, QUADRANTS
from itchi.declaratorias_6h import municipio_pixels
from itchi.mswep import subset_mswep_domain
from itchi.pipeline import compute_itchi_snapshot_from_grid
from itchi.plotting import SIAT_DANGER_LABELS, classify_itchi_to_siat_levels, get_siat_hazard_cmap_norm
from itchi.precipitation import (
    RAMP_R4_NODES,
    compute_antecedent_precipitation,
    compute_precipitation_hazard_ramp,
    compute_precipitation_matrix_hazard,
)
from itchi.tracks import DEFAULT_R34_COLUMN_MAP

ROOT = get_project_root()
CFG = load_default_config()
print("itchi", ITCHI_VERSION, "|", ROOT)

In [ ]:
CASES = {
    "CRISTOBAL2020": dict(name="CRISTOBAL", season=2020, basin="NA", atcf="AL032020", t_map="2020-06-04 00:00"),
    "ODILE2014": dict(name="ODILE", season=2014, basin="EP", atcf="EP152014", t_map="2014-09-15 00:00"),
}


def local_path(path) -> Path:
    # "D:/..." -> "/mnt/d/..." en WSL/Linux; sin cambios en Windows
    s = str(path).replace("\\", "/")
    if os.name != "nt" and re.match(r"^[A-Za-z]:/", s):
        return Path(f"/mnt/{s[0].lower()}") / s[3:]
    return Path(s)


MSWEP_6H_DIR = local_path("D:/MSWEP_V316_6H")
CLIM = {d: ROOT / "data" / "processed" / f"mswep_v3_{d}h_percentiles_may_nov.nc" for d in (6, 24, 72)}
CLIM_VAR = "p_wet_raw"
CLIM_DOMAIN = dict(lon_min=-140.0, lon_max=-60.0, lat_min=5.0, lat_max=35.0)
PAD_DEG = 12.0

RAMPS = {"R3": {"Q90": 0.0, "Q95": 0.5, "Q99": 1.0}, "R4": RAMP_R4_NODES}
EXPERIMENTS = {                                   # los que pasan por el pipeline
    "v0.1": dict(precip="R3", wind="v01"),
    "R4": dict(precip="R4", wind="v01"),
    "R4+esc": dict(precip="matriz", wind="v01"),
    "v0.2": dict(precip="matriz", wind="w06_tail"),
}
TAG = {"v0.1": "v01", "R4": "R4", "R4+esc": "R4esc", "v0.2": "v02"}   # nombres en el NetCDF
CHAIN = ["v0.1", "R4", "R4+esc", "W06 sin cola", "v0.2"]
STEPS = ["rampa R4", "escalada", "viento W06 en R34", "cola de viento"]
STORE = ("ITCHI", "H_P", "H_W", "H_dir")

EXP_STYLE = {
    "v0.1": dict(color="#6B7280", ls="--"), "R4": dict(color="#E8A25A", ls=":"),
    "R4+esc": dict(color="#8E5CC2", ls="-."), "W06 sin cola": dict(color="#7FB3E6", ls="-"),
    "v0.2": dict(color="#2A6FC9", ls="-"),
}
STEP_COLORS = ["#E8A25A", "#8E5CC2", "#2A9D8F", "#D9822B"]

K_ALERT = (2, 3, 4)            # la clase 1 queda fuera: con la cola actual cubre toda la zona
SNAP_OFFSETS_H = (-12, 0, 12, 24)   # snapshots de los mapas, relativos a t_map
LOCAL_OFFSET_H = 6             # fecha_inicio de la declaratoria a 00 h local (UTC-6) = 06 UTC
FORCE_RECOMPUTE = False        # True: recalcula aunque exista el NetCDF
SAVE_FRAMES = False            # True: un PNG por snapshot (v0.1 contra v0.2) para animar

DECL_FILE = ROOT / "outputs" / "diagnostics" / "declaratorias_6h" / "declaratorias_ct_eventos.csv"
MUN_SHP = ROOT / "data" / "external" / "mun23gw" / "mun23gw.shp"
CACHE_DIR = ROOT / "outputs" / "diagnostics" / "itchi_v02_snapshots"
FIG_DIR = ROOT / "outputs" / "figures" / "itchi_v02_snapshots"
CSV_05 = ROOT / "outputs" / "figures" / "itchi_v02_mejoras" / "verificacion_municipal_casos.csv"
for d in (CACHE_DIR, FIG_DIR):
    d.mkdir(parents=True, exist_ok=True)

assert MSWEP_6H_DIR.is_dir(), f"no existe {MSWEP_6H_DIR}"
for d, f in CLIM.items():
    assert f.exists(), f"falta {f.name}: corre scripts/percentiles_6h.py --durations {d}"
SIAT_CMAP, SIAT_NORM = get_siat_hazard_cmap_norm()
PC = ccrs.PlateCarree()


def clase(x) -> np.ndarray:
    # Clase SIAT 0–5 en tramos de 0.2 (0 = sin peligro)
    return np.asarray(classify_itchi_to_siat_levels(np.nan_to_num(np.asarray(x, dtype=float)))).astype(np.int8)


print("configuración OK | caché:", CACHE_DIR, "| figuras:", FIG_DIR)

## 1. Lectura de datos y cálculo de ITCHI por snapshot

Mismas funciones que el notebook 05: trayectoria IBTrACS con radios PA21, lluvia observada de MSWEP V3.16 ($P_6$, $R_{24}$, $A_{72}$) y percentiles de cada duración. La diferencia es que aquí se guardan todos los snapshots y los cuatro experimentos que pasan por el pipeline.

In [ ]:
def load_track(case: str) -> pd.DataFrame:
    c = CASES[case]
    ibtracs = ROOT / CFG["paths"]["tracks"] / f"IBTrACS.{c['basin']}.v04r01.nc"
    rocloud = ROOT / "data" / "external" / "Rocloud" / f"ROCLOUD_{c['basin']}.csv"
    track = H.load_case_track(ibtracs, rocloud, c["name"], c["season"], c["atcf"])
    ok = np.array([H.attribution_radii_km(row, "pa21")[0] is not None for _, row in track.iterrows()])
    return track[ok].reset_index(drop=True)


def case_bbox(track: pd.DataFrame) -> dict:
    return dict(
        lon_min=max(float(track["lon"].min()) - PAD_DEG, CLIM_DOMAIN["lon_min"]),
        lon_max=min(float(track["lon"].max()) + PAD_DEG, CLIM_DOMAIN["lon_max"]),
        lat_min=max(float(track["lat"].min()) - PAD_DEG, CLIM_DOMAIN["lat_min"]),
        lat_max=min(float(track["lat"].max()) + PAD_DEG, CLIM_DOMAIN["lat_max"]),
    )


def read_precip(track: pd.DataFrame, bbox: dict):
    # P6, R24 y A72 observados en los tiempos de la trayectoria
    t = pd.DatetimeIndex(track["time"])
    path = MSWEP_6H_DIR / f"mswep_v3_6h_{t[0].year}.nc"
    with xr.open_dataset(path) as ds:
        p6 = subset_mswep_domain(ds["precipitation"], **bbox)
        p6 = p6.sel(time=slice(t.min() - pd.Timedelta(hours=84), t.max())).load()
    missing = t.difference(pd.DatetimeIndex(p6["time"].values))
    assert missing.empty, f"tiempos sin acumulado de 6 h: {list(missing)}"
    r24 = compute_antecedent_precipitation(p6, hours=24, exclude_current=False)
    a72 = compute_antecedent_precipitation(p6, hours=72, exclude_current=True)
    return tuple(da.sel(time=t.values) for da in (p6, r24, a72))


def read_clim(d: int, template: xr.DataArray) -> dict:
    with xr.open_dataset(CLIM[d]) as ds:
        q = ds[CLIM_VAR].load()
    q = q.assign_coords(quantile=[f"Q{v * 100:g}" for v in q["quantile"].values])
    q = q.reindex_like(template, method="nearest", tolerance=1e-3)
    return {k: q.sel(quantile=k, drop=True) for k in q["quantile"].values}


def precip_hazards(p6, r24, a72, q6, q24, q72) -> dict:
    hp = {
        "R3": compute_precipitation_hazard_ramp(p6, q6, RAMPS["R3"]),
        "R4": compute_precipitation_hazard_ramp(p6, q6, RAMPS["R4"]),
        "matriz": compute_precipitation_matrix_hazard(p6, q6, {"R24": r24, "A72": a72}, {"R24": q24, "A72": q72}),
    }
    return {k: v.fillna(0.0).transpose("time", "lat", "lon") for k, v in hp.items()}

In [ ]:
def compute_case(case: str, track: pd.DataFrame, bbox: dict) -> xr.Dataset:
    p6, r24, a72 = read_precip(track, bbox)
    template = p6.isel(time=0, drop=True)
    q6, q24, q72 = (read_clim(d, template) for d in (6, 24, 72))
    hp = precip_hazards(p6, r24, a72, q6, q24, q72)
    lat, lon = template["lat"].values, template["lon"].values
    lon2d, lat2d = np.meshgrid(lon, lat)

    data, masks = {}, {"M_direct": [], "M_indirect": []}
    print(f"  {case}: lluvia y percentiles listos, malla {lon2d.shape}, {len(track)} snapshots", flush=True)
    for exp, spec in EXPERIMENTS.items():
        t0 = time.time()
        fields = {k: [] for k in STORE}
        for i, row in track.iterrows():
            radii, _ = H.attribution_radii_km(row, "pa21")
            rmw = float(row["rmw_km"]) if np.isfinite(row["rmw_km"]) else float(row["pa21_rmax_km"])
            fallback = float(row["pa21_r34_km"]) if np.isfinite(row["pa21_r34_km"]) else None
            res = compute_itchi_snapshot_from_grid(
                p6.values[i], None, None, None,
                precipitation_hazard=hp[spec["precip"]].values[i],
                lon=lon2d, lat=lat2d, center_lon=float(row["lon"]), center_lat=float(row["lat"]),
                r34_by_quadrant={q: row[DEFAULT_R34_COLUMN_MAP[q]] for q in QUADRANTS}, r34_unit="nm",
                rocloud_by_quadrant=radii, rocloud_unit="km",
                vmax_kt=float(row["vmax_kt"]), rmw_km=rmw, fallback_direct_radius_km=fallback,
                wind_model=spec["wind"], run_quality_control=True,
            )
            for k in STORE:
                fields[k].append(np.asarray(res[k], dtype=np.float32))
            for m in masks:                       # las regiones no dependen del experimento
                mk = np.asarray(res[m], dtype=bool)
                if exp == "v0.1":
                    masks[m].append(mk)
                else:
                    assert np.array_equal(mk, masks[m][i]), f"{m} distinto en {exp}, paso {i}"
        for k, v in fields.items():
            data[f"{k}__{TAG[exp]}"] = (("time", "lat", "lon"), np.stack(v))
        print(f"  {case} · {exp}: {time.time() - t0:.0f} s", flush=True)
    for m, v in masks.items():
        data[m] = (("time", "lat", "lon"), np.stack(v).astype(np.int8))
    data["P6"] = (("time", "lat", "lon"), p6.values.astype(np.float32))
    ds = xr.Dataset(data, coords={"time": track["time"].values, "lat": lat, "lon": lon})
    ds.attrs.update(case=case, itchi_version=ITCHI_VERSION)
    return ds


def load_case(case: str):
    track = load_track(case)
    bbox = case_bbox(track)
    f = CACHE_DIR / f"{case.lower()}_snapshots.nc"
    ds = None
    if f.exists() and not FORCE_RECOMPUTE:
        print(f"{case}: leyendo {f.name}", flush=True)
        ds = xr.load_dataset(f)
        if not np.array_equal(pd.DatetimeIndex(ds["time"].values), pd.DatetimeIndex(track["time"])):
            warnings.warn(f"{f.name} no coincide con la trayectoria: se recalcula")
            ds = None
    if ds is None:
        print(f"{case}: calculando 4 experimentos (solo la primera vez)", flush=True)
        ds = compute_case(case, track, bbox)
        t0 = time.time()
        ds.to_netcdf(f, encoding={v: {"zlib": True, "complevel": 1} for v in ds.data_vars})
        print(f"  guardado {f.name} ({f.stat().st_size / 1e6:.0f} MB, {time.time() - t0:.0f} s)", flush=True)
    return track, bbox, ds


def cell_area_km2(lat2d, lon2d):
    dlat = abs(float(np.diff(lat2d[:2, 0])[0])); dlon = abs(float(np.diff(lon2d[0, :2])[0]))
    return (np.radians(dlat) * H.EARTH_RADIUS_KM) * (np.radians(dlon) * H.EARTH_RADIUS_KM * np.cos(np.radians(lat2d)))


def unpack(track, bbox, ds) -> dict:
    lat, lon = ds["lat"].values, ds["lon"].values
    lon2d, lat2d = np.meshgrid(lon, lat)
    F = {e: {k: ds[f"{k}__{TAG[e]}"].values for k in STORE} for e in EXPERIMENTS}
    m_dir, m_ind = ds["M_direct"].values.astype(bool), ds["M_indirect"].values.astype(bool)
    v2 = F["v0.2"]
    F["W06 sin cola"] = {"ITCHI": 1 - (1 - v2["H_dir"]) * (1 - v2["H_P"] * m_ind),
                         "H_P": v2["H_P"], "H_W": v2["H_W"] * m_dir, "H_dir": v2["H_dir"]}
    return dict(track=track, bbox=bbox, times=pd.DatetimeIndex(ds["time"].values), lon2d=lon2d, lat2d=lat2d,
                area=cell_area_km2(lat2d, lon2d), F={e: F[e] for e in CHAIN},
                M_direct=m_dir, M_indirect=m_ind, P6=ds["P6"].values)


CASE_DATA = {}
for case in CASES:
    CASE_DATA[case] = unpack(*load_case(case))
    C = CASE_DATA[case]
    print(f"{case}: {len(C['times'])} snapshots, {C['times'][0]:%Y-%m-%d %H} a {C['times'][-1]:%Y-%m-%d %H} UTC, malla {C['lon2d'].shape}")

**Comprobaciones de construcción.** La rampa R4 nunca baja el peligro respecto a R3. La escalada nunca lo baja respecto a R4. La cola nunca lo baja respecto a `W06 sin cola`. El cambio de viento dentro de R34 puede ir en los dos sentidos: W06 calibrado contra Rankine. Si una de las tres primeras falla, hay un error en los campos.

In [ ]:
for case, C in CASE_DATA.items():
    F = C["F"]
    for a, b in (("v0.1", "R4"), ("R4", "R4+esc"), ("W06 sin cola", "v0.2")):
        dmin = float(np.nanmin(F[b]["ITCHI"] - F[a]["ITCHI"]))
        assert dmin > -1e-5, f"{case}: {b} < {a} (mín {dmin:.2e})"
    d = F["W06 sin cola"]["ITCHI"] - F["R4+esc"]["ITCHI"]
    print(f"{case}: monotonía OK | viento W06 en R34 − Rankine: de {np.nanmin(d):+.2f} a {np.nanmax(d):+.2f}")

## 2. Municipios, tierra y declaratorias

- **Universo.** Municipios con algún píxel en la región de atribución, en cualquier snapshot. Es el mismo universo que en el notebook 05.
- **Valor municipal por snapshot.** El máximo de ITCHI sobre los píxeles del municipio en ese snapshot, y su clase SIAT. Se obtiene una matriz municipios × tiempo por experimento.
- **Tierra.** Los píxeles dentro de algún municipio. Las áreas de la sección 4 se miden solo ahí, porque el peligro sobre el mar no se puede verificar con declaratorias.
- **Declaratorias.** $D = 1$ si el municipio tuvo declaratoria asociada al ciclón (`CT_pico`). Para cada municipio con declaratoria se guarda la fecha de inicio más temprana (a las 00 h locales, 06 UTC) y el tiempo del pico de lluvia municipal (`t_pico`).

In [ ]:
gdf = gpd.read_file(MUN_SHP)
gdf["CVEGEO"] = gdf["CVEGEO"].astype(str).str.zfill(5)
gdf = gdf.to_crs(4326) if gdf.crs is not None and gdf.crs.to_epsg() != 4326 else gdf
decl = pd.read_csv(DECL_FILE, dtype={"CVEGEO": str})


def municipal_max_series(field: np.ndarray, flats: list) -> np.ndarray:
    # (T, ny, nx) -> (municipios, T): máximo sobre los píxeles de cada municipio
    flat = np.nan_to_num(field.reshape(field.shape[0], -1))
    starts = np.cumsum([0] + [len(f) for f in flats[:-1]])
    return np.maximum.reduceat(flat[:, np.concatenate(flats)], starts, axis=1).T


def municipal_setup(case: str, C: dict) -> dict:
    b = C["bbox"]
    rp = gdf.geometry.representative_point()
    sub = gdf[rp.x.between(b["lon_min"], b["lon_max"]) & rp.y.between(b["lat_min"], b["lat_max"])]
    pix = municipio_pixels(sub, C["lat2d"][:, 0], C["lon2d"][0], sub["CVEGEO"])
    region = (C["M_direct"] | C["M_indirect"]).any(axis=0).ravel()
    land = np.zeros(region.size, bool)
    for flat, n in pix.values():
        if n > 0:
            land[flat] = True
    codes = [c for c, (flat, _) in pix.items() if region[flat].any()]
    flats = [pix[c][0] for c in codes]

    d = decl[decl["CT_pico"] == CASES[case]["atcf"]]
    inicio = pd.to_datetime(d["fecha_inicio"]).groupby(d["CVEGEO"]).min() + pd.Timedelta(hours=LOCAL_OFFSET_H)
    pico = pd.to_datetime(d["t_pico"]).groupby(d["CVEGEO"]).min()
    idx = pd.Index(codes)
    vals = {e: municipal_max_series(C["F"][e]["ITCHI"], flats) for e in CHAIN}
    return dict(
        codes=np.array(codes), land=land.reshape(C["lon2d"].shape),
        D=idx.isin(d["CVEGEO"]), D_desastre=idx.isin(d.loc[d["Tipo Declaratoria"] == "Desastre", "CVEGEO"]),
        t_inicio=inicio.reindex(idx).to_numpy("datetime64[ns]"), t_pico=pico.reindex(idx).to_numpy("datetime64[ns]"),
        vals=vals, cls={e: clase(v) for e, v in vals.items()},
    )


MUNI = {}
for case, C in CASE_DATA.items():
    MUNI[case] = M = municipal_setup(case, C)
    fuera = set(decl.loc[decl["CT_pico"] == CASES[case]["atcf"], "CVEGEO"]) - set(M["codes"])
    print(f"{case}: {len(M['codes'])} municipios en la zona | con declaratoria {M['D'].sum()} | desastre {M['D_desastre'].sum()}"
          f" | municipios con declaratoria fuera de la zona: {len(fuera)}")

## 3. Verificación: funciones y comprobación contra el notebook 05

Son las mismas métricas del notebook 06. POD es la fracción de municipios con declaratoria que tuvieron alerta. POFD es la fracción sin declaratoria que la tuvieron. PSS = POD − POFD. AUC es la probabilidad de que un municipio con declaratoria tenga un valor mayor que uno sin ella.

**Acumulado hasta $t$.** La clase acumulada de un municipio en el snapshot $t$ es la clase máxima que alcanzó desde el inicio hasta $t$. Al final del evento coincide con la clase de `ITCHI_max`. Por eso el PSS acumulado final debe ser igual al del notebook 05. La segunda celda lo comprueba.

In [ ]:
def pod_pofd(alert: np.ndarray, obs: np.ndarray) -> tuple:
    pod = (alert & obs).sum() / max(obs.sum(), 1)
    pofd = (alert & ~obs).sum() / max((~obs).sum(), 1)
    return pod, pofd, pod - pofd


def auc(score: np.ndarray, obs: np.ndarray) -> float:
    # Mann–Whitney por rangos; los empates cuentan la mitad
    r = pd.Series(score).rank().to_numpy()
    n1, n0 = obs.sum(), (~obs).sum()
    return float((r[obs].sum() - n1 * (n1 + 1) / 2) / (n1 * n0)) if n1 and n0 else np.nan


def first_alert(cls: np.ndarray, k: int) -> np.ndarray:
    # Índice del primer snapshot con clase >= k; T si nunca
    hit = cls >= k
    return np.where(hit.any(axis=1), hit.argmax(axis=1), cls.shape[1])


def chain_table(case: str, target: str = "D") -> pd.DataFrame:
    # Verificación del evento (clase de ITCHI_max) para cada experimento de la cadena
    M = MUNI[case]
    obs = M[target]; keep = ~M["D"] | obs        # con target = desastre, se quitan los otros positivos
    rows = []
    for e in CHAIN:
        cmax = M["cls"][e].max(axis=1)
        for k in range(1, 6):
            pod, pofd, pss = pod_pofd((cmax >= k)[keep], obs[keep])
            rows.append({"caso": case, "evento": target, "experimento": e, "clase ≥": k, "POD": pod, "POFD": pofd, "PSS": pss})
    return pd.DataFrame(rows)


CADENA = pd.concat([chain_table(c, t) for c in CASES for t in ("D", "D_desastre")], ignore_index=True)
CADENA.to_csv(FIG_DIR / "cadena_verificacion_evento.csv", index=False, float_format="%.3f")

In [ ]:
if CSV_05.exists():
    s05 = pd.read_csv(CSV_05)
    s05 = s05[s05["evento"] == "cualquier declaratoria"].rename(columns={"clase >=": "clase ≥"})
    m = CADENA[CADENA["evento"] == "D"].merge(s05, on=["caso", "experimento", "clase ≥"], suffixes=("", "_05"))
    print(f"comparados {len(m)} valores con el notebook 05 | máx. |ΔPSS| = {np.abs(m['PSS'] - m['PSS_05']).max():.4f}")
else:
    print("no está la tabla del notebook 05; se omite la comparación")

## 4. Mapas por snapshot

Filas: los cinco experimentos de la cadena. Columnas: snapshots alrededor de `t_map`. Contorno negro: región directa (R34). Contorno gris: región de atribución (R_W06). Estrella: centro del ciclón.

In [ ]:
def map_axes(C: dict, nrows: int, ncols: int, size=(4.2, 3.4)):
    fig, axes = plt.subplots(nrows, ncols, figsize=(size[0] * ncols, size[1] * nrows),
                             subplot_kw={"projection": PC}, squeeze=False)
    b = C["bbox"]
    for ax in axes.ravel():
        ax.set_extent([b["lon_min"], b["lon_max"], b["lat_min"], b["lat_max"]], crs=PC)
        ax.coastlines(lw=0.5)
        ax.plot(C["track"]["lon"], C["track"]["lat"], "k-", lw=0.6, transform=PC)
    return fig, axes


def regions(ax, C: dict, j: int):
    ax.contour(C["lon2d"], C["lat2d"], C["M_direct"][j].astype(float), levels=[0.5], colors="k", linewidths=0.8, transform=PC)
    ax.contour(C["lon2d"], C["lat2d"], (C["M_direct"][j] | C["M_indirect"][j]).astype(float), levels=[0.5],
               colors="0.45", linewidths=0.8, linestyles="--", transform=PC)
    row = C["track"].iloc[j]
    ax.plot(row["lon"], row["lat"], "k*", ms=9, transform=PC)


def snapshot_times(case: str) -> list:
    t0 = pd.Timestamp(CASES[case]["t_map"]); times = CASE_DATA[case]["times"]
    return [t0 + pd.Timedelta(hours=h) for h in SNAP_OFFSETS_H if t0 + pd.Timedelta(hours=h) in times]


def siat_masked(x):
    lv = clase(x).astype(float)
    return np.where(lv > 0, lv, np.nan)


for case, C in CASE_DATA.items():
    ts = snapshot_times(case)
    fig, axes = map_axes(C, len(CHAIN), len(ts))
    for c, t in enumerate(ts):
        j = C["times"].get_loc(t)
        for r, e in enumerate(CHAIN):
            ax = axes[r, c]
            m = ax.pcolormesh(C["lon2d"], C["lat2d"], siat_masked(C["F"][e]["ITCHI"][j]), cmap=SIAT_CMAP,
                              norm=SIAT_NORM, transform=PC, shading="auto")
            regions(ax, C, j)
            if r == 0:
                ax.set_title(f"{t:%m-%d %H} UTC", fontsize=11)
            if c == 0:
                ax.text(-0.08, 0.5, e, transform=ax.transAxes, rotation=90, va="center", ha="right", fontsize=11)
    cb = fig.colorbar(m, ax=axes.ravel().tolist(), shrink=0.4, ticks=np.arange(6) + 0.5)
    cb.set_ticklabels(SIAT_DANGER_LABELS)
    fig.suptitle(f"{case}: clase SIAT por snapshot", fontsize=13)
    fig.savefig(FIG_DIR / f"{case.lower()}_snapshots_cadena.png", dpi=170, bbox_inches="tight")

### Qué paso de la cadena cambia cada píxel

En los mismos snapshots:
- **Fila superior:** cambio de clase de v0.2 respecto a v0.1.
- **Fila inferior:** el paso de la cadena con el mayor cambio de clase en cada píxel (rampa R4, escalada, viento W06 dentro de R34 o cola de viento).

Así se separa, por ejemplo, si el aumento de clase alta en Odile viene de la lluvia o del perfil de viento.

In [ ]:
CAUSE_CMAP = ListedColormap(STEP_COLORS)
CAUSE_NORM = BoundaryNorm(np.arange(0.5, len(STEPS) + 1), len(STEPS))


def cause_fields(C: dict, j: int):
    cl = [clase(C["F"][e]["ITCHI"][j]) for e in CHAIN]
    d = np.stack([b.astype(int) - a.astype(int) for a, b in zip(cl[:-1], cl[1:])])
    cause = np.where(np.abs(d).max(axis=0) > 0, np.abs(d).argmax(axis=0) + 1, 0)
    return cl[-1].astype(int) - cl[0].astype(int), cause


for case, C in CASE_DATA.items():
    ts = snapshot_times(case)
    fig, axes = map_axes(C, 2, len(ts))
    for c, t in enumerate(ts):
        j = C["times"].get_loc(t)
        total, cause = cause_fields(C, j)
        m1 = axes[0, c].pcolormesh(C["lon2d"], C["lat2d"], np.where(total != 0, total, np.nan), cmap="RdBu_r",
                                   vmin=-3.5, vmax=3.5, transform=PC, shading="auto")
        axes[1, c].pcolormesh(C["lon2d"], C["lat2d"], np.where(cause > 0, cause, np.nan), cmap=CAUSE_CMAP,
                              norm=CAUSE_NORM, transform=PC, shading="auto")
        for ax in axes[:, c]:
            regions(ax, C, j)
        axes[0, c].set_title(f"{t:%m-%d %H} UTC", fontsize=11)
    fig.colorbar(m1, ax=axes[0].tolist(), shrink=0.8, ticks=range(-3, 4), label="Δ clase (v0.2 − v0.1)")
    fig.legend(handles=[Patch(color=c_, label=s) for c_, s in zip(STEP_COLORS, STEPS)], loc="lower center",
               ncol=len(STEPS), frameon=False, fontsize=10)
    fig.suptitle(f"{case}: cambio de clase y paso que lo produce", fontsize=13)
    fig.savefig(FIG_DIR / f"{case.lower()}_snapshots_causa.png", dpi=170, bbox_inches="tight")

## 5. Métricas por snapshot

Para cada snapshot y experimento se calcula:
- **ITCHI máximo en tierra:** el peor valor sobre México en ese momento.
- **Área en tierra por clase:** área (10³ km², celdas ponderadas por $\cos\varphi$) con clase ≥ k. Mide la extensión del peligro; las clases ≥ 3 corresponden a ITCHI > 0.4.
- **$\overline{H_P}$ y $\overline{H_W}$ en la región de atribución:** separan el aporte de lluvia y de viento.
- **Municipios en alerta:** número de municipios con clase ≥ k en ese snapshot.
- **PSS instantáneo:** verificación usando solo la clase del snapshot. Sube cuando el peligro está sobre los municipios afectados, y baja cuando cubre municipios que nunca tuvieron declaratoria.
- **PSS acumulado:** verificación con la clase máxima hasta ese snapshot. Muestra cuándo se construye la discriminación del evento. Al final es igual al PSS del evento.

**Por qué importa.** Dos versiones con el mismo `ITCHI_max` pueden ser muy distintas como alerta: una puede marcar los municipios antes, o mantener la alerta solo mientras dura el peligro. Estas series muestran ese comportamiento, que el máximo del evento oculta.

In [ ]:
def snapshot_table(case: str) -> pd.DataFrame:
    C, M = CASE_DATA[case], MUNI[case]
    land, area = M["land"], C["area"]
    region = C["M_direct"] | C["M_indirect"]
    rows = []
    for e in CHAIN:
        f = C["F"][e]
        cls_m = M["cls"][e]
        acum = np.maximum.accumulate(cls_m, axis=1)
        for j, t in enumerate(C["times"]):
            lv = clase(f["ITCHI"][j][land])
            reg = region[j]
            r = {"caso": case, "experimento": e, "time": t,
                 "ITCHI máx. en tierra": float(np.nanmax(f["ITCHI"][j][land])),
                 "H_P medio en la región": float(np.nanmean(f["H_P"][j][reg])) if reg.any() else np.nan,
                 "H_W medio en la región": float(np.nanmean(f["H_W"][j][reg])) if reg.any() else np.nan}
            for k in range(1, 6):
                r[f"área tierra clase ≥ {k} (10³ km²)"] = float(area[land][lv >= k].sum() / 1e3)
            for k in K_ALERT:
                r[f"municipios clase ≥ {k}"] = int((cls_m[:, j] >= k).sum())
                r[f"PSS instantáneo ≥ {k}"] = pod_pofd(cls_m[:, j] >= k, M["D"])[2]
                r[f"PSS acumulado ≥ {k}"] = pod_pofd(acum[:, j] >= k, M["D"])[2]
            rows.append(r)
    return pd.DataFrame(rows)


SNAP = pd.concat([snapshot_table(c) for c in CASES], ignore_index=True)
SNAP.to_csv(FIG_DIR / "metricas_por_snapshot.csv", index=False, float_format="%.4f")
SNAP.groupby(["caso", "experimento"], sort=False)[["ITCHI máx. en tierra", "área tierra clase ≥ 3 (10³ km²)",
                                                    "área tierra clase ≥ 5 (10³ km²)", "municipios clase ≥ 3"]].mean().round(2)

In [ ]:
def ts_axes(nrows: int, ncols: int):
    fig, axes = plt.subplots(nrows, ncols, figsize=(6.2 * ncols, 2.4 * nrows), sharex="col", squeeze=False)
    for ax in axes[-1]:
        loc = mdates.AutoDateLocator(); ax.xaxis.set_major_locator(loc)
        ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
    return fig, axes


PANELS = [("ITCHI máx. en tierra", "ITCHI máx.\nen tierra"),
          ("área tierra clase ≥ 3 (10³ km²)", "área clase ≥ 3\n(10³ km²)"),
          ("área tierra clase ≥ 5 (10³ km²)", "área clase 5\n(10³ km²)"),
          ("H_P medio en la región", "H_P medio"),
          ("H_W medio en la región", "H_W medio"),
          ("municipios clase ≥ 3", "municipios\nclase ≥ 3")]
fig, axes = ts_axes(len(PANELS), len(CASES))
for c, case in enumerate(CASES):
    for e in CHAIN:
        d = SNAP[(SNAP["caso"] == case) & (SNAP["experimento"] == e)]
        for r, (col, lab) in enumerate(PANELS):
            axes[r, c].plot(d["time"], d[col], lw=1.8, label=e, **EXP_STYLE[e])
            axes[r, 0].set_ylabel(lab, fontsize=9)
    axes[0, c].set_title(case)
axes[0, -1].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "metricas_por_snapshot.png", dpi=200, bbox_inches="tight")

**Cómo leer.**
- La rampa R4 y la escalada solo cambian $\overline{H_P}$.
- El viento W06 y la cola solo cambian $\overline{H_W}$.
- Si el área de clase 5 de la v0.2 crece en los mismos snapshots en que crece $\overline{H_W}$, el aumento viene del viento y no de la lluvia.

In [ ]:
fig, axes = ts_axes(2 * len(K_ALERT), len(CASES))
for c, case in enumerate(CASES):
    for e in CHAIN:
        d = SNAP[(SNAP["caso"] == case) & (SNAP["experimento"] == e)]
        for i, k in enumerate(K_ALERT):
            axes[2 * i, c].plot(d["time"], d[f"PSS instantáneo ≥ {k}"], lw=1.6, label=e, **EXP_STYLE[e])
            axes[2 * i + 1, c].plot(d["time"], d[f"PSS acumulado ≥ {k}"], lw=1.6, label=e, **EXP_STYLE[e])
            axes[2 * i, 0].set_ylabel(f"PSS inst.\nclase ≥ {k}", fontsize=9)
            axes[2 * i + 1, 0].set_ylabel(f"PSS acum.\nclase ≥ {k}", fontsize=9)
    axes[0, c].set_title(case)
    for ax in axes[:, c]:
        ax.axhline(0, color="0.6", lw=0.6)
axes[0, -1].legend(frameon=False, fontsize=8, ncol=2)
fig.tight_layout()
fig.savefig(FIG_DIR / "pss_por_snapshot.png", dpi=200, bbox_inches="tight")

**Cómo leer.**
- El PSS instantáneo es bajo al inicio porque la lluvia todavía no llega a los municipios afectados. Su máximo marca el momento en que el peligro coincide mejor con los municipios con declaratoria.
- El PSS acumulado nunca baja por POD, pero sí puede bajar cuando la alerta cubre municipios sin declaratoria.
- Una versión que llega antes a su valor final discrimina más temprano en el evento.

## 6. Atribución por paso de la cadena

### 6.1 A nivel evento

PSS del evento (clase de `ITCHI_max`) para cada experimento de la cadena y su cambio en cada paso. Responde, por ejemplo, de dónde viene la caída de PSS en clases 4–5 de Odile en el notebook 05.

In [ ]:
def delta_steps(cad: pd.DataFrame) -> pd.DataFrame:
    p = cad.pivot_table(index=["caso", "clase ≥"], columns="experimento", values="PSS")[CHAIN]
    out = pd.DataFrame({s: p[b] - p[a] for s, a, b in zip(STEPS, CHAIN[:-1], CHAIN[1:])})
    out["total (v0.2 − v0.1)"] = p["v0.2"] - p["v0.1"]
    return out


cad = CADENA[CADENA["evento"] == "D"]
print("PSS del evento por experimento")
display(cad.pivot_table(index=["caso", "clase ≥"], columns="experimento", values="PSS")[CHAIN].round(3))
DELTA = delta_steps(cad)
DELTA.to_csv(FIG_DIR / "cadena_delta_pss.csv", float_format="%.3f")
print("Cambio de PSS en cada paso")
DELTA.round(3)

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(6.2 * len(CASES), 3.6), sharey=True, squeeze=False)
w = 0.2
for ax, case in zip(axes[0], CASES):
    d = DELTA.loc[case]
    for i, s in enumerate(STEPS):
        ax.bar(d.index + (i - 1.5) * w, d[s], w, color=STEP_COLORS[i], label=s, edgecolor="white")
    ax.plot(d.index, d["total (v0.2 − v0.1)"], "k_", ms=18, mew=2, label="total")
    ax.axhline(0, color="0.4", lw=0.8)
    ax.set(xticks=range(1, 6), xlabel="alerta si clase SIAT ≥", ylabel="Δ PSS", title=case)
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "cadena_delta_pss.png", dpi=200, bbox_inches="tight")

**Cómo leer.** Barras arriba de cero: ese paso mejora la discriminación en esa clase. La raya negra es el cambio total.
- La rampa R4 y la cola de viento actúan sobre todo en las clases bajas.
- La escalada y el viento W06 actúan en las altas.
- En Odile, con 6 municipios con declaratoria, un municipio vale 0.17 de POD: solo los cambios grandes y consistentes con Cristóbal merecen atención.

### 6.2 Por snapshot

Número de municipios cuya clase sube (arriba) o baja (abajo) en cada paso de la cadena, en cada snapshot. La tabla siguiente separa los que suben según tengan o no declaratoria. Un paso útil sube sobre todo municipios con declaratoria; uno que sube en proporción a la tasa base no agrega información.

In [ ]:
def step_changes(case: str) -> pd.DataFrame:
    M, times = MUNI[case], CASE_DATA[case]["times"]
    rows = []
    for s, a, b in zip(STEPS, CHAIN[:-1], CHAIN[1:]):
        d = M["cls"][b].astype(int) - M["cls"][a].astype(int)
        for j, t in enumerate(times):
            up, dn = d[:, j] > 0, d[:, j] < 0
            rows.append({"caso": case, "paso": s, "time": t, "suben": int(up.sum()), "bajan": int(dn.sum()),
                         "suben con declaratoria": int((up & M["D"]).sum()), "suben sin declaratoria": int((up & ~M["D"]).sum())})
    return pd.DataFrame(rows)


CAMBIOS = pd.concat([step_changes(c) for c in CASES], ignore_index=True)
CAMBIOS.to_csv(FIG_DIR / "cambios_por_paso_snapshot.csv", index=False)
res = CAMBIOS.groupby(["caso", "paso"], sort=False)[["suben", "bajan", "suben con declaratoria", "suben sin declaratoria"]].sum()
res["% de subidas con declaratoria"] = 100 * res["suben con declaratoria"] / res["suben"].where(res["suben"] > 0)
res["tasa base (%)"] = [100 * MUNI[c]["D"].mean() for c, _ in res.index]
res.round(1)

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(6.2 * len(CASES), 3.6), squeeze=False)
for ax, case in zip(axes[0], CASES):
    d = CAMBIOS[CAMBIOS["caso"] == case]
    times = CASE_DATA[case]["times"]
    width = 0.8 * (times[1] - times[0]) if len(times) > 1 else pd.Timedelta(hours=5)
    up = np.zeros(len(times)); dn = np.zeros(len(times))
    for i, s in enumerate(STEPS):
        v = d[d["paso"] == s]
        ax.bar(times, v["suben"], width, bottom=up, color=STEP_COLORS[i], label=s)
        ax.bar(times, -v["bajan"].to_numpy(), width, bottom=-dn, color=STEP_COLORS[i], alpha=0.55)
        up += v["suben"].to_numpy(); dn += v["bajan"].to_numpy()
    ax.axhline(0, color="k", lw=0.6)
    ax.set(title=case, ylabel="municipios (suben + / bajan −)")
    loc = mdates.AutoDateLocator(); ax.xaxis.set_major_locator(loc)
    ax.xaxis.set_major_formatter(mdates.ConciseDateFormatter(loc))
axes[0, 0].legend(frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "cambios_por_paso_snapshot.png", dpi=200, bbox_inches="tight")

## 7. La alerta a lo largo del evento en los municipios con declaratoria

Cada fila es un municipio con declaratoria y cada columna un snapshot. El color es la clase SIAT de ese municipio en ese snapshot. Los municipios están ordenados por el primer snapshot en que la v0.2 alcanza clase ≥ 3. El punto negro marca el pico de lluvia municipal (`t_pico`).

Muestra en una sola figura cuándo empieza la alerta, cuánto dura y si llega antes o después del pico de lluvia.

In [ ]:
def plot_hovmoller(case: str, exps=("v0.1", "v0.2"), k_sort: int = 3):
    M, times = MUNI[case], CASE_DATA[case]["times"]
    pos = np.where(M["D"])[0]
    if pos.size == 0:
        return None
    ref = M["cls"]["v0.2"][pos]
    order = pos[np.lexsort((-ref.max(axis=1), first_alert(ref, k_sort)))]
    T = len(times)
    jp = np.array([np.argmin(np.abs(times.values - tp)) if not np.isnat(tp) else -1 for tp in M["t_pico"][order]])
    fig, axes = plt.subplots(1, len(exps), figsize=(5.2 * len(exps), 1.6 + 0.045 * len(order)), sharey=True, squeeze=False)
    step = max(1, T // 8)
    for ax, e in zip(axes[0], exps):
        img = M["cls"][e][order].astype(float)
        img[img == 0] = np.nan
        ax.imshow(img, aspect="auto", cmap=SIAT_CMAP, norm=SIAT_NORM, interpolation="nearest")
        ok = jp >= 0
        ax.plot(jp[ok], np.arange(len(order))[ok], "k.", ms=2.5)
        ax.set_xlim(-0.5, T - 0.5); ax.set_ylim(len(order) - 0.5, -0.5)
        ax.set_xticks(range(0, T, step))
        ax.set_xticklabels([f"{t:%m-%d %H}" for t in times[::step]], rotation=45, ha="right", fontsize=8)
        ax.set_title(e)
    axes[0, 0].set_ylabel(f"municipios con declaratoria (n = {len(order)})")
    axes[0, 0].set_yticks([])
    fig.suptitle(f"{case}: clase SIAT municipal por snapshot", fontsize=12)
    fig.tight_layout()
    return fig


for case in CASES:
    fig = plot_hovmoller(case)
    if fig is not None:
        fig.savefig(FIG_DIR / f"{case.lower()}_hovmoller_municipios.png", dpi=200, bbox_inches="tight")

## 8. Duración y oportunidad de la alerta

**Duración.** Horas que cada municipio pasa con clase ≥ k: número de snapshots × 6 h. Se compara su AUC con el de la clase máxima del evento. Si la duración discrimina mejor que el máximo, conviene reportar la persistencia de la alerta además del pico.

**Oportunidad.** Para los municipios con declaratoria, se miden las horas entre la primera alerta (clase ≥ k) y dos referencias:
- la fecha de inicio de la declaratoria, a 00 h locales;
- el pico de lluvia municipal.

Positivo significa que la alerta llegó antes. ITCHI se calcula aquí con lluvia observada: esto no es tiempo de anticipación de un pronóstico. Dice qué tan temprano en la evolución del evento el índice marca al municipio. La fecha de inicio es diaria y administrativa, así que el error es de ± 12 h.

In [ ]:
def duration_table(case: str) -> pd.DataFrame:
    M, times = MUNI[case], CASE_DATA[case]["times"]
    dt_h = float(np.median(np.diff(times.values)) / np.timedelta64(1, "h")) if len(times) > 1 else 6.0
    rows = []
    for e in CHAIN:
        cls = M["cls"][e]
        r = {"caso": case, "experimento": e, "AUC clase máx.": auc(cls.max(axis=1), M["D"])}
        for k in K_ALERT:
            dur = dt_h * (cls >= k).sum(axis=1)
            r[f"AUC horas ≥ {k}"] = auc(dur, M["D"])
            r[f"mediana h ≥ {k} con decl."] = float(np.median(dur[M["D"]])) if M["D"].any() else np.nan
            r[f"mediana h ≥ {k} sin decl."] = float(np.median(dur[~M["D"]]))
        rows.append(r)
    return pd.DataFrame(rows)


def lead_table(case: str) -> pd.DataFrame:
    M, times = MUNI[case], CASE_DATA[case]["times"]
    T, pos = len(times), M["D"]
    rows = []
    for e in CHAIN:
        for k in K_ALERT:
            fi = first_alert(M["cls"][e], k)
            alerted = fi < T
            t_first = np.full(len(fi), np.datetime64("NaT"), dtype="datetime64[ns]")
            t_first[alerted] = times.values[fi[alerted]]
            r = {"caso": case, "experimento": e, "clase ≥": k, "detectados (%)": 100 * alerted[pos].mean() if pos.any() else np.nan}
            for ref, lab in (("t_inicio", "inicio"), ("t_pico", "pico de lluvia")):
                lead = (M[ref] - t_first) / np.timedelta64(1, "h")
                sel = pos & alerted & ~np.isnan(lead)
                r[f"mediana h antes del {lab}"] = float(np.median(lead[sel])) if sel.any() else np.nan
                r[f"% antes del {lab}"] = 100 * float((lead[sel] > 0).mean()) if sel.any() else np.nan
            rows.append(r)
    return pd.DataFrame(rows)


DURACION = pd.concat([duration_table(c) for c in CASES], ignore_index=True)
OPORTUNIDAD = pd.concat([lead_table(c) for c in CASES], ignore_index=True)
DURACION.to_csv(FIG_DIR / "duracion_alerta.csv", index=False, float_format="%.3f")
OPORTUNIDAD.to_csv(FIG_DIR / "oportunidad_alerta.csv", index=False, float_format="%.1f")
display(DURACION.round(3))
OPORTUNIDAD.round(1)

In [ ]:
fig, axes = plt.subplots(1, len(CASES), figsize=(6.2 * len(CASES), 3.6), sharey=True, squeeze=False)
k = 3
for ax, case in zip(axes[0], CASES):
    M, times = MUNI[case], CASE_DATA[case]["times"]
    dt_h = float(np.median(np.diff(times.values)) / np.timedelta64(1, "h")) if len(times) > 1 else 6.0
    data, pos_x = [], []
    for i, e in enumerate(CHAIN):
        dur = dt_h * (M["cls"][e] >= k).sum(axis=1)
        data += [dur[M["D"]], dur[~M["D"]]]; pos_x += [3 * i, 3 * i + 1]
    bp = ax.boxplot(data, positions=pos_x, widths=0.8, showfliers=False, patch_artist=True,
                    medianprops=dict(color="k", lw=1.5))
    for j, patch in enumerate(bp["boxes"]):
        patch.set(facecolor=EXP_STYLE[CHAIN[j // 2]]["color"], alpha=1.0 if j % 2 == 0 else 0.35)
    ax.set_xticks([3 * i + 0.5 for i in range(len(CHAIN))])
    ax.set_xticklabels(CHAIN, rotation=20, fontsize=9)
    ax.set(title=f"{case}: horas con clase ≥ {k}", ylabel="horas")
axes[0, 0].legend(handles=[Patch(color="0.4", label="con declaratoria"), Patch(color="0.4", alpha=0.35, label="sin declaratoria")],
                  frameon=False, fontsize=8)
fig.tight_layout()
fig.savefig(FIG_DIR / "duracion_alerta.png", dpi=200, bbox_inches="tight")

## 9. Cuadros para animación (opcional)

Con `SAVE_FRAMES = True` se guarda un PNG por snapshot, v0.1 contra v0.2, en `outputs/figures/itchi_v02_snapshots/frames_<caso>/`. Con esos cuadros se puede armar un GIF o un video para la presentación.

In [ ]:
if SAVE_FRAMES:
    for case, C in CASE_DATA.items():
        out = FIG_DIR / f"frames_{case.lower()}"
        out.mkdir(exist_ok=True)
        for j, t in enumerate(C["times"]):
            fig, axes = map_axes(C, 1, 2, size=(5.2, 4.2))
            for ax, e in zip(axes[0], ("v0.1", "v0.2")):
                ax.pcolormesh(C["lon2d"], C["lat2d"], siat_masked(C["F"][e]["ITCHI"][j]), cmap=SIAT_CMAP, norm=SIAT_NORM,
                              transform=PC, shading="auto")
                regions(ax, C, j)
                ax.set_title(f"{e} | {t:%Y-%m-%d %H} UTC", fontsize=10)
            fig.savefig(out / f"{t:%Y%m%d%H}.png", dpi=110, bbox_inches="tight")
            plt.close(fig)
        print(case, "cuadros en", out)

## 10. Resumen por caso y experimento

In [ ]:
rows = []
for case in CASES:
    for e in CHAIN:
        s = SNAP[(SNAP["caso"] == case) & (SNAP["experimento"] == e)]
        c = CADENA[(CADENA["caso"] == case) & (CADENA["experimento"] == e) & (CADENA["evento"] == "D")].set_index("clase ≥")
        d = DURACION[(DURACION["caso"] == case) & (DURACION["experimento"] == e)].iloc[0]
        o = OPORTUNIDAD[(OPORTUNIDAD["caso"] == case) & (OPORTUNIDAD["experimento"] == e) & (OPORTUNIDAD["clase ≥"] == 3)].iloc[0]
        rows.append({"caso": case, "experimento": e,
                     "área media clase ≥ 3 por snapshot (10³ km²)": s["área tierra clase ≥ 3 (10³ km²)"].mean(),
                     "máx. municipios clase ≥ 3": s["municipios clase ≥ 3"].max(),
                     **{f"PSS evento ≥ {k}": c.loc[k, "PSS"] for k in K_ALERT},
                     "AUC clase máx.": d["AUC clase máx."], "AUC horas ≥ 3": d["AUC horas ≥ 3"],
                     "mediana h antes del inicio (≥ 3)": o["mediana h antes del inicio"]})
RESUMEN = pd.DataFrame(rows)
RESUMEN.to_csv(FIG_DIR / "resumen_snapshots.csv", index=False, float_format="%.3f")
print("figuras y tablas en", FIG_DIR)
RESUMEN.round(3)